In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px


In [ ]:
def fibonacci_sphere( n: int):
        k = np.arange(n) + 0.5
        z = 1 - 2 * k / n
        r = np.sqrt(np.clip(1 - z * z, 0.0, 1.0))
        phi = (np.pi * (1 + 5**0.5)) * k  # golden angle progression
        x, y = r * np.cos(phi), r * np.sin(phi)
        return np.stack([x, y, z], axis=1)

def n_dirs_for_degree( dir_deg):
    # Target geodesic spacing (radians)
    alpha = np.deg2rad(max(0.5, float(dir_deg)))  # guard against too-small/zero
    # For small alpha, spherical cap area π α^2 ≈ 4π/N  =>  N ≈ (2/α)^2
    n = int(np.ceil((2.0 / alpha) ** 2))
    return max(n, 6)  # tiny floor to avoid degeneracy


In [ ]:
deg = 60
n = n_dirs_for_degree(deg)
v = fibonacci_sphere(n)

c = v[0, :]
print(c)

import scipy

R = scipy.spatial.transform.Rotation.from_euler('xyz', [60, 0, 0], degrees=False)
R = R.as_matrix()

u = R @ c
print(u)

# compute the inner product of u and v
inner_product = u @ v.T
print(inner_product)

k = int(np.argmax(inner_product))
print(k)





In [ ]:
# Interactive 3D visualization with plotly
def visualize_fibonacci_sphere_interactive(points, deg, show_sphere=True):
    """
    Create an interactive 3D visualization of Fibonacci sphere points
    
    Args:
        points: numpy array of shape (n, 3) containing the 3D points
        deg: degree parameter used to generate the points
        show_sphere: whether to show the unit sphere as reference
    """
    
    # Create scatter plot for the Fibonacci sphere points
    scatter = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1], 
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=8,
            color='red',
            symbol='circle',
            opacity=0.8,
            line=dict(width=2, color='darkred')
        ),
        name=f'Fibonacci Points (n={len(points)}, deg={deg}°)',
        hovertemplate='<b>Point %{pointNumber}</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    data = [scatter]
    
    # Add unit sphere if requested
    if show_sphere:
        # Create a unit sphere mesh
        u = np.linspace(0, 2 * np.pi, 50)
        v = np.linspace(0, np.pi, 50)
        x_sphere = np.outer(np.cos(u), np.sin(v))
        y_sphere = np.outer(np.sin(u), np.sin(v))
        z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
        
        sphere = go.Surface(
            x=x_sphere, y=y_sphere, z=z_sphere,
            opacity=0.1,
            colorscale='Blues',
            showscale=False,
            name='Unit Sphere'
        )
        data.append(sphere)
    
    # Create the figure
    fig = go.Figure(data=data)
    
    # Update layout for better visualization
    fig.update_layout(
        title=f'Interactive Fibonacci Sphere Visualization (n={len(points)}, deg={deg}°)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y', 
            zaxis_title='Z',
            aspectmode='cube',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        width=800,
        height=600,
        showlegend=True
    )
    
    return fig

# Create and display the interactive visualization
fig = visualize_fibonacci_sphere_interactive(v, deg, show_sphere=True)
fig.show()


In [ ]:
# Let's also create a comparison with different degree settings
degrees_to_compare = [30, 45, 60, 90]

figs = []
for deg_comp in degrees_to_compare:
    n_comp = n_dirs_for_degree(deg_comp)
    v_comp = fibonacci_sphere(n_comp)
    
    fig = visualize_fibonacci_sphere_interactive(v_comp, deg_comp, show_sphere=True)
    fig.update_layout(title=f'Degree {deg_comp}° (n={n_comp})')
    figs.append(fig)

# Display the first comparison (degree 30)
print("Comparison for degree 30°:")
for fig in figs:
    fig.show()


In [ ]:
# Additional analysis and interactive features
def analyze_fibonacci_sphere(points):
    """Analyze the distribution properties of Fibonacci sphere points"""
    
    # Calculate distances between consecutive points
    distances = []
    for i in range(len(points)):
        for j in range(i+1, len(points)):
            dist = np.linalg.norm(points[i] - points[j])
            distances.append(dist)
    
    distances = np.array(distances)
    
    print(f"Number of points: {len(points)}")
    print(f"Min distance between points: {distances.min():.4f}")
    print(f"Max distance between points: {distances.max():.4f}")
    print(f"Mean distance between points: {distances.mean():.4f}")
    print(f"Std distance between points: {distances.std():.4f}")
    
    return distances

# Analyze the current Fibonacci sphere
print("Analysis for current sphere (degree 60°):")
distances = analyze_fibonacci_sphere(v)

# Create a histogram of distances
fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=distances,
    nbinsx=20,
    name='Point Distances',
    marker_color='lightblue',
    opacity=0.7
))

fig_hist.update_layout(
    title='Distribution of Distances Between Fibonacci Sphere Points',
    xaxis_title='Distance',
    yaxis_title='Frequency',
    width=600,
    height=400
)

fig_hist.show()


In [ ]:
# Visualize the sampled points, initial vector, and rotated vector
def visualize_vectors_and_points(points, initial_vector, rotated_vector, rotation_angle_deg):
    """
    Create an interactive 3D visualization showing:
    - Fibonacci sphere points
    - Initial vector (c)
    - Rotated vector (u)
    - Unit sphere reference
    """
    
    # Create scatter plot for the Fibonacci sphere points
    scatter_points = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1], 
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color='lightblue',
            symbol='circle',
            opacity=0.6,
            line=dict(width=1, color='blue')
        ),
        name='Fibonacci Points',
        hovertemplate='<b>Point %{pointNumber}</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    # Create initial vector (c)
    initial_vector_line = go.Scatter3d(
        x=[0, initial_vector[0]],
        y=[0, initial_vector[1]],
        z=[0, initial_vector[2]],
        mode='lines+markers',
        line=dict(color='red', width=8),
        marker=dict(size=10, color='red'),
        name='Initial Vector (c)',
        hovertemplate='<b>Initial Vector</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    # Create rotated vector (u)
    rotated_vector_line = go.Scatter3d(
        x=[0, rotated_vector[0]],
        y=[0, rotated_vector[1]],
        z=[0, rotated_vector[2]],
        mode='lines+markers',
        line=dict(color='green', width=8),
        marker=dict(size=10, color='green'),
        name=f'Rotated Vector (u, {rotation_angle_deg:.1f}°)',
        hovertemplate='<b>Rotated Vector</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    # Create unit sphere mesh
    u_sphere = np.linspace(0, 2 * np.pi, 30)
    v_sphere = np.linspace(0, np.pi, 30)
    x_sphere = np.outer(np.cos(u_sphere), np.sin(v_sphere))
    y_sphere = np.outer(np.sin(u_sphere), np.sin(v_sphere))
    z_sphere = np.outer(np.ones(np.size(u_sphere)), np.cos(v_sphere))
    
    sphere = go.Surface(
        x=x_sphere, y=y_sphere, z=z_sphere,
        opacity=0.1,
        colorscale='Blues',
        showscale=False,
        name='Unit Sphere'
    )
    
    # Create the figure
    fig = go.Figure(data=[scatter_points, initial_vector_line, rotated_vector_line, sphere])
    
    # Update layout for better visualization
    fig.update_layout(
        title=f'Fibonacci Sphere Points with Vectors (Rotation: {rotation_angle_deg:.1f}°)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y', 
            zaxis_title='Z',
            aspectmode='cube',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

# Create the visualization
rotation_angle_deg = 60  # This should match your rotation
fig_vectors = visualize_vectors_and_points(v, c, u, rotation_angle_deg)
fig_vectors.show()


In [ ]:
# Enhanced visualization highlighting the closest point
def visualize_with_closest_point(points, initial_vector, rotated_vector, closest_point_idx, inner_products, rotation_angle_deg):
    """
    Create an interactive 3D visualization highlighting the closest point
    """
    
    # Create scatter plot for all Fibonacci sphere points
    scatter_all = go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1], 
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color='lightblue',
            symbol='circle',
            opacity=0.6,
            line=dict(width=1, color='blue')
        ),
        name='Fibonacci Points',
        hovertemplate='<b>Point %{pointNumber}</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<br>' +
                      'Inner Product: %{customdata:.3f}<extra></extra>',
        customdata=inner_products
    )
    
    # Highlight the closest point
    scatter_closest = go.Scatter3d(
        x=[points[closest_point_idx, 0]],
        y=[points[closest_point_idx, 1]],
        z=[points[closest_point_idx, 2]],
        mode='markers',
        marker=dict(
            size=15,
            color='yellow',
            symbol='diamond',
            opacity=1.0,
            line=dict(width=3, color='orange')
        ),
        name=f'Closest Point (idx={closest_point_idx})',
        hovertemplate='<b>Closest Point</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<br>' +
                      f'Inner Product: {inner_products[closest_point_idx]:.3f}<extra></extra>'
    )
    
    # Create initial vector (c)
    initial_vector_line = go.Scatter3d(
        x=[0, initial_vector[0]],
        y=[0, initial_vector[1]],
        z=[0, initial_vector[2]],
        mode='lines+markers',
        line=dict(color='red', width=8),
        marker=dict(size=10, color='red'),
        name='Initial Vector (c)',
        hovertemplate='<b>Initial Vector</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    # Create rotated vector (u)
    rotated_vector_line = go.Scatter3d(
        x=[0, rotated_vector[0]],
        y=[0, rotated_vector[1]],
        z=[0, rotated_vector[2]],
        mode='lines+markers',
        line=dict(color='green', width=8),
        marker=dict(size=10, color='green'),
        name=f'Rotated Vector (u, {rotation_angle_deg:.1f}°)',
        hovertemplate='<b>Rotated Vector</b><br>' +
                      'X: %{x:.3f}<br>' +
                      'Y: %{y:.3f}<br>' +
                      'Z: %{z:.3f}<extra></extra>'
    )
    
    # Create unit sphere mesh
    u_sphere = np.linspace(0, 2 * np.pi, 30)
    v_sphere = np.linspace(0, np.pi, 30)
    x_sphere = np.outer(np.cos(u_sphere), np.sin(v_sphere))
    y_sphere = np.outer(np.sin(u_sphere), np.sin(v_sphere))
    z_sphere = np.outer(np.ones(np.size(u_sphere)), np.cos(v_sphere))
    
    sphere = go.Surface(
        x=x_sphere, y=y_sphere, z=z_sphere,
        opacity=0.1,
        colorscale='Blues',
        showscale=False,
        name='Unit Sphere'
    )
    
    # Create the figure
    fig = go.Figure(data=[scatter_all, scatter_closest, initial_vector_line, rotated_vector_line, sphere])
    
    # Update layout for better visualization
    fig.update_layout(
        title=f'Fibonacci Sphere with Vectors - Closest Point Highlighted (Rotation: {rotation_angle_deg:.1f}°)',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y', 
            zaxis_title='Z',
            aspectmode='cube',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

# Create the enhanced visualization
fig_closest = visualize_with_closest_point(v, c, u, k, inner_product, rotation_angle_deg)
fig_closest.show()

# Print some analysis
print(f"Initial vector c: {c}")
print(f"Rotated vector u: {u}")
print(f"Closest point index: {k}")
print(f"Closest point coordinates: {v[k]}")
print(f"Maximum inner product: {inner_product[k]:.4f}")
print(f"All inner products: {inner_product}")
